In [2]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!unzip "/content/drive/MyDrive/AlbumGAN/Data/cleaned_dataset.zip"

Archive:  /content/drive/MyDrive/AlbumGAN/Data/cleaned_dataset.zip
replace audio/92720046.mp3? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

Pulled all the code directly from here: https://github.com/LAION-AI/CLAP

You might need to set up a Hugging Face token if you don't already have one. You can make an account on Hugging Face, get a read-only token using a button on the profile thing, and then paste your token into the 'Secrets' tab (the little key icon on the left) in this Colab.

In [ ]:
!pip install laion-clap

In [ ]:
import numpy as np
import librosa
import torch
import laion_clap

In [ ]:
# quantization
def int16_to_float32(x):
    return (x / 32767.0).astype('float32')


def float32_to_int16(x):
    x = np.clip(x, a_min=-1., a_max=1.)
    return (x * 32767.).astype('int16')

model = laion_clap.CLAP_Module(enable_fusion=False)
model.load_ckpt() # download the default pretrained checkpoint.

In [ ]:
# Directly get audio embeddings from audio files
audio_file = [
    '/content/audio/100357738.mp3',
]
audio_embed = model.get_audio_embedding_from_filelist(x = audio_file, use_tensor=True)
print(audio_embed)
print(audio_embed.shape)

In [ ]:
import pandas as pd

df = pd.read_csv('/content/cleaned_dataset.csv')
print(len(df))

In [ ]:
audio_files = ['/content/audio/' + df.iloc[i]['audio_filename'] for i in range(len(df))]

embeddings = model.get_audio_embedding_from_filelist(x = audio_files, use_tensor=True)

In [ ]:
track_ids = df['deezer_id'].astype(str).tolist()

# save as a dictionary so that the ids never get shuffled
embedding_dict = {t_id: emb for t_id, emb in zip(track_ids, embeddings_matrix)}
torch.save(embedding_dict, "/content/embeddings/audio_embeddings.pt")